# Challenge 2 · The Entanglement Detective
**Tier 1 - Foundations** · **Estimated time: 20-30 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Two qubits always give matching answers. Is that entanglement - or just two coins that were set to the same face? Collect the evidence like a detective.

### Learning objectives
- Construct a Bell state and read its joint measurement outcomes.
- Show that correlation in a single basis is not proof of entanglement.
- Measure correlations in several bases (Z, X, Y).
- Decide entanglement of a pure two-qubit state from the purity of one qubit.

### Prerequisites
- Challenge 1 or equivalent: gates, exact simulation, probabilities
- The H and CNOT gates

### What you will write
| Function | Status |
|---|---|
| `bell_circuit` | required |
| `joint_probabilities` | required |
| `correlation_in_basis` | required |
| `is_entangled` | required |
| `chsh_value` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `bell_circuit()` prepares |Phi+> = (|00>+|11>)/sqrt(2) (up to global phase) with unitary gates only.
- Independent superposition (H on both qubits) is not accepted: in the Bell state each qubit alone is maximally mixed.
- `joint_probabilities` returns the exact probability of all 2^n outcomes in Qiskit bit order.
- `correlation_in_basis` equals <sigma x sigma> for sigma in {Z, X, Y} on arbitrary two-qubit states.
- `is_entangled` separates entangled from product states, including perfectly Z-correlated product states.
- Inputs that are not two-qubit states, and unknown bases, raise ValueError; the tolerance is respected.

**Bonus (optional, never required to pass):**
- `chsh_value` reproduces the CHSH quantity (2*sqrt(2) for the Bell state, at most 2 for product states).

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
Two qubits always give matching answers when measured. Is that **entanglement** - or two coins that were simply set to the same face?
Collect the evidence like a detective:

1. **Build** the Bell state $|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt2$.
2. **Analyse the joint outcomes** exactly: the probability of every bit-string.
3. **Measure correlations in more than one basis** (Z, X, Y).
4. **Decide** whether a pure two-qubit state is entangled - and do not be fooled by suspects that look entangled in one basis.
5. *(Bonus)* Compute the CHSH quantity.

### Function contracts
States are Qiskit `Statevector` objects. **Qubit 0 is the least significant bit** (the rightmost character of a bit-string).

| Function | Input | Output | Must raise `ValueError` when |
|---|---|---|---|
| `bell_circuit()` | none | 2-qubit `QuantumCircuit`, gates only (no measurement), preparing &#124;Φ+⟩ from &#124;00⟩ up to a global phase | - |
| `joint_probabilities(state)` | a `Statevector` of qubits | `dict` mapping **every** bit-string of length n (including impossible ones) to its exact probability (plain Python `str` keys and `float` values) | the argument is not a `Statevector` of qubits |
| `correlation_in_basis(state, basis)` | 2-qubit `Statevector`; `basis` is `"Z"`, `"X"` or `"Y"` | float E = P(equal outcomes) − P(different outcomes) when **both** qubits are measured in that basis (outcome bit 0 = eigenvalue +1) | not a 2-qubit state, or `basis` not in `{"Z","X","Y"}` |
| `is_entangled(state, tol=1e-9)` | pure 2-qubit `Statevector` | `bool`: True iff 1 − purity of qubit 0's reduced state exceeds `tol` | not a 2-qubit state |
| *(bonus)* `chsh_value(state)` | 2-qubit `Statevector` | S = ⟨A0B0⟩+⟨A0B1⟩+⟨A1B0⟩−⟨A1B1⟩ with Alice = qubit 0, Bob = qubit 1, A0 = Z, A1 = X, B0 = (Z+X)/√2, B1 = (Z−X)/√2 | not a 2-qubit state |

**Tolerances.** Exact comparisons use 1e-9. `tol=1e-9` in `is_entangled` also protects against floating-point noise: a *product* state has purity 1 only up to ~1e-16.

## Concept: coins versus entanglement
* A **product state** $|a\rangle\otimes|b\rangle$ gives *independent* outcomes in every basis. Each qubit alone is in a pure state.
* An **entangled** state cannot be written that way. For $|\Phi^+\rangle$ each qubit *alone* is completely random (maximally mixed), yet the pair is perfectly correlated.
* **One basis is not enough.** $|00\rangle$ is perfectly Z-correlated but not entangled. Entanglement shows up as correlation that survives a *change of basis*: $|\Phi^+\rangle$ is correlated in Z **and** in X.
* **For a pure two-qubit state** the reduced state of one qubit is pure **iff** the state is a product. Its *purity* ${\rm Tr}\,\rho_A^2$ is 1 for products and 1/2 for a maximally entangled state.
* Beware of the *phase trap*: $\tfrac12(|00\rangle+|01\rangle+|10\rangle-|11\rangle)$ has perfectly uniform, independent-looking Z statistics - but it is entangled.

To measure in the **X basis** apply H to a qubit and then measure Z. For the **Y basis** apply S† and then H.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace

In [ ]:
from qiskit_fest.challenges.c02_entanglement import plot_case_file, suspects
from qiskit_fest.viz import draw_circuit, plot_distributions

CASE_FILE = suspects()
print("Suspects:", list(CASE_FILE))

## Part 1 · The Bell state  *(TODO)*
Complete `bell_circuit()`: a **two-qubit** circuit that prepares $(|00\rangle+|11\rangle)/\sqrt2$ from $|00\rangle$. Use gates only - no measurements.
Remember the trap: two *independent* superpositions (H on both qubits) is **not** entanglement.

<details><summary>💡 Hint 1 (subtle)</summary>

Start by putting one qubit into superposition. Then make the second qubit *depend* on the first.
</details>

<details><summary>💡 Hint 2</summary>

The controlled-NOT gate (`cx(control, target)`) flips the target when the control is |1⟩. What does it do to (|0⟩+|1⟩)|0⟩ / √2?
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

One Hadamard on qubit 0, then one CNOT with qubit 0 as control and qubit 1 as target.
</details>

In [ ]:
def bell_circuit() -> QuantumCircuit:
    """Two-qubit circuit (gates only) preparing (|00> + |11>)/sqrt(2) from |00>."""
    qc = QuantumCircuit(2)
    # TODO: superposition on one qubit, then entangle the other qubit with it.
    raise NotImplementedError("TODO: implement bell_circuit()")

In [ ]:
with if_ready():
    qc = bell_circuit()
    draw_circuit(qc, "Your Bell circuit")

In [ ]:
report = run_public_checks(2, globals(), keyword="bell_circuit")

## Part 2 · Joint outcomes  *(TODO)*
Complete `joint_probabilities(state)`: the exact probability of every bit-string, **including the impossible ones (probability 0)**.
Keys are bit-strings with qubit 0 on the right, so for two qubits the keys are `"00"`, `"01"`, `"10"`, `"11"`.
Raise `ValueError` if the argument is not a `Statevector` of qubits.

<details><summary>💡 Hint 1 (subtle)</summary>

A `Statevector` knows its own outcome probabilities. Check how it orders them (which index is which bit-string?).
</details>

<details><summary>💡 Hint 2</summary>

`state.probabilities()` returns a vector whose entry *i* belongs to the bit-string with integer value *i*. `format(i, f"0{n}b")` turns *i* into an *n*-bit string.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Validate the type and `state.dims()` (all 2), then build a dict over `range(2**n)` with plain `float` values.
</details>

In [ ]:
def joint_probabilities(state: Statevector) -> dict[str, float]:
    """Exact probability of every n-bit string (qubit 0 on the right), zeros included. ValueError if not a qubit Statevector."""
    # TODO: validate the argument and build the dictionary.
    raise NotImplementedError("TODO: implement joint_probabilities()")

In [ ]:
with if_ready():
    bell_state = Statevector(bell_circuit())
    probs = joint_probabilities(bell_state)
    print(probs)
    plot_distributions({"your Bell state": probs}, "Joint outcomes of the Bell state")

In [ ]:
report = run_public_checks(2, globals(), keyword="joint_probabilities")

## Part 3 · Correlation in a chosen basis  *(TODO)*
Complete `correlation_in_basis(state, basis)`. Measure **both** qubits in the Z, X or Y basis and return
$$E = P(\text{equal outcomes}) - P(\text{different outcomes}) = p_{00}+p_{11}-p_{01}-p_{10}.$$
Outcome bit 0 means eigenvalue +1. For a perfectly correlated pair E = +1, for a perfectly anti-correlated pair E = −1, for independent fair coins E = 0.

*Basis change:* to measure in X, apply `h` to each qubit and then read the Z probabilities; for Y, apply `sdg` then `h` to each qubit.
`Statevector.evolve(circuit)` applies a circuit to a state. Raise `ValueError` for a state that is not two-qubit or for an unknown basis.

<details><summary>💡 Hint 1 (subtle)</summary>

"Measuring in the X basis" = "rotating the state so that X eigenstates become Z eigenstates, then measuring Z".
</details>

<details><summary>💡 Hint 2</summary>

Build a small two-qubit circuit with the basis-change gates, evolve the state with it, then reuse your `joint_probabilities`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Z needs no rotation; X: `h` on both qubits; Y: `sdg` then `h` on both qubits. Finally combine the four probabilities with the formula above.
</details>

In [ ]:
def correlation_in_basis(state: Statevector, basis: str) -> float:
    """E = P(equal) - P(different) when both qubits are measured in `basis` ('Z', 'X' or 'Y'). ValueError on bad input."""
    # TODO: validate, rotate into the measurement basis, combine the joint probabilities.
    raise NotImplementedError("TODO: implement correlation_in_basis()")

In [ ]:
with if_ready():
    print("Bell state: ", {b: round(correlation_in_basis(Statevector(bell_circuit()), b), 3) for b in "ZXY"})
    print("Both zero : ", {b: round(correlation_in_basis(CASE_FILE["C: both zero"], b), 3) for b in "ZXY"})

In [ ]:
report = run_public_checks(2, globals(), keyword="correlation")

## Part 4 · The verdict  *(TODO)*
Complete `is_entangled(state, tol=1e-9)` for a **pure** two-qubit state: return `True` iff $1-\mathrm{Tr}\rho_A^2 > \text{tol}$, where $\rho_A$ is the reduced state of qubit 0.
Use `partial_trace(state, [1])` (trace out qubit 1) and the purity of the result. The `tol` matters: product states have purity 1 only up to ~1e-16.

<details><summary>💡 Hint 1 (subtle)</summary>

If you look at only one qubit of a product state, what do you see? And of a Bell state? "How mixed" is a number.
</details>

<details><summary>💡 Hint 2</summary>

`partial_trace(state, [1])` returns a `DensityMatrix` for qubit 0; it has a `purity()` method (the result may be a complex number with zero imaginary part - take the real part).
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`purity` < 1 means mixed means entangled. Compare `1 - purity` with `tol` - do **not** compare `purity < 1` directly (rounding noise would call product states entangled).
</details>

In [ ]:
def is_entangled(state: Statevector, tol: float = 1e-9) -> bool:
    """True iff the pure two-qubit state is entangled (1 - purity of qubit 0's reduced state > tol). ValueError otherwise."""
    # TODO: validate, take the reduced state of one qubit, compare its purity with 1.
    raise NotImplementedError("TODO: implement is_entangled()")

In [ ]:
with if_ready():
    plot_case_file(CASE_FILE, correlation_in_basis, is_entangled)

In [ ]:
report = run_public_checks(2, globals(), keyword="is_entangled")

## Bonus · The CHSH quantity  *(optional)*
For measurement settings A0 = Z, A1 = X (Alice, qubit 0) and B0 = (Z+X)/√2, B1 = (Z−X)/√2 (Bob, qubit 1) the CHSH quantity is
$S=\langle A_0B_0\rangle+\langle A_0B_1\rangle+\langle A_1B_0\rangle-\langle A_1B_1\rangle$.
Any classical (local hidden-variable) model has $|S|\le 2$; the Bell state reaches $2\sqrt2\approx2.83$.
Complete `chsh_value(state)`. *(Hint: with these settings the cross terms cancel and S = √2 (⟨ZZ⟩ + ⟨XX⟩).)*

In [ ]:
def chsh_value(state: Statevector) -> float:
    """CHSH quantity S for the settings A0=Z, A1=X, B0=(Z+X)/sqrt2, B1=(Z-X)/sqrt2 (Alice = qubit 0, Bob = qubit 1)."""
    # TODO (bonus): combine the correlations you already computed.
    raise NotImplementedError("TODO (bonus): implement chsh_value()")

In [ ]:
with if_ready():
    print("S(Bell)         =", round(chsh_value(Statevector(bell_circuit())), 4), "(classical bound 2, quantum maximum 2.8284)")
    print("S(two coins)    =", round(chsh_value(CASE_FILE["A: two coins"]), 4))

In [ ]:
report = run_public_checks(2, globals(), keyword="bonus")

## Reflection
1. State **A** (two coins) and state **B** both look random if you measure only one qubit. What measurement tells them apart?
2. State **C** is perfectly Z-correlated. Why is it *not* entangled?
3. State **E** has independent-looking Z statistics. How did your `is_entangled` still catch it?
4. Why does a tolerance appear in `is_entangled`? What could go wrong with `purity < 1`?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 2)